## LAsset replication, Step 1: closed set → occurrence profiles → relationships (15 test modules)

The 15 test modules are the `data/RTL_data` modules that have manual GT: bus, cache, cpu, cpu_cp_cfu, cpu_cp_muldiv, cpu_pmp, debug_dtm, hwspinlock, imem, spi, sys, trng, twi, uart, wdt.

| stage | who | what |
|---|---|---|
| 1a closed set | code | `rtl_parse.parse_rtl_file`, with record types from every RTL file. Before writing, a self-test: boot_rom, trng and cache re-extracted must equal the element lists the profiler was tuned on, and boot_rom must match its hand-read lines. |
| 1b occurrence profiles | **v3d** profiler | v3b's cells, run with only the module list, closed-set folder and output folder swapped. extract + structure by code; classify and validate by gpt-5-mini (medium); batches of at most 20 elements and 85 occurrences. **Rulebook v3d `c748e39bfc05`** (`occurrence_prompts_v3d/`): v3c's two fixes (arrays of records, function arguments) plus the parentheses rules found in three astra rounds; prompts classify `97ef25455d47`, validate `021e62bcd902`. **Validate's CODE CHECK** also gets two facts computed by code (`position_facts`): which occurrence left of `<=` is the target, and whether a right-hand side is exactly one element (DIRR_ASS). |
| 1c relationships | relation annotator | `relation_stage` with the prompt in `relation_prompts_v1/relation_system_prompt.md`, on the 1b profiles. |

Why v3d: see `occurrence_prompts_v3d/RULEBOOK_CHANGELOG.md`. On the 2,142 test entries, distinct entries flagged by the code checks (`step1/tag_regress.py`): v3b 26, v3c 30 and 48 (the same prompt run twice), v3d 3. The same prompt run twice changes about 2% of entries by itself.

**Where every JSON file is saved** (all under `step1/lasset_step1/`):

| folder | contents |
|---|---|
| `closed_set/<module>.json` | the closed set: `{"module", "source", "entities": [{"entity", "ports": [{name, dir, type}], "signals": [{name, type}]}]}` |
| `occurrence_profiles/extract/<code sha>/<module>__<entity>__bNN.json` | the occurrence inventory with Context and Path (code) |
| `occurrence_profiles/classify/97ef25455d47/...` | SITE and Role per occurrence (gpt-5-mini) |
| `occurrence_profiles/validate/021e62bcd902/...` | the checked occurrence profiles, used by 1c |
| `relation_out/<prompt sha>/<module>__<entity>__bNN.json` | the relation annotator's raw answers, with usage |
| `relation_map/<prompt sha>/<module>.json` | **the final traceable map**, `{"ports": [...], "signals": [...]}`, one file per module |

Earlier profiler versions stay on disk in their own sha folders (v3b `cb5e024b7d65`, v3c `0180458a45fb`); the test rounds are in `rounds/`.

**How to run.** Each run cell plans without calling the API when its switch is `"none"`. Answers are cached by prompt and input sha, so a re-run only calls what is missing.
1. Run 1a (no API; it also reloads the code).
2. 1b: set `PROFILE_ON = "all"`. The 32 test batches of the last round are reused where their input is unchanged.
3. Run the check cell.
4. 1c: set `RELATE_ON` to one module first, then `"all"`. Only modules whose profiles are complete are sent.


In [6]:
# Step 1a: the closed set of the 15 test modules, extracted by code, after a self-test (no API call)
import importlib, sys
sys.path.insert(0, "step1")
import lasset_step1 as S1, relation_stage as RS
importlib.reload(RS); importlib.reload(S1)

TEST15 = S1.test_modules()
assert S1.closed_set_selftest(), "the extractor does not reproduce the tuned element lists: stop here"
for m, (ne, np_, ns) in S1.build_closed_set(TEST15).items():
    print(f"   {m:24s} entities {ne}, ports {np_:>4}, signals {ns:>3}")
print("written to", S1.CLOSED)


   neorv32_boot_rom: re-extracted closed set equals the tuned element list (21 elements, 1 entities)
   neorv32_trng: re-extracted closed set equals the tuned element list (56 elements, 3 entities)
   neorv32_cache: re-extracted closed set equals the tuned element list (86 elements, 2 entities)
   neorv32_boot_rom against the hand-read lines 21-24, 38-39: PASS (ports [('clk_i', 'in', 'std_ulogic'), ('rstn_i', 'in', 'std_ulogic'), ('bus_req_i', 'in', 'bus_req_t'), ('bus_rsp_o', 'out', 'bus_rsp_t')], signals [('rden', 'std_ulogic'), ('rdata', 'std_ulogic_vector')], 15 record fields)
   closed-set self-test: PASS
   neorv32_bus              entities 6, ports  811, signals  59
   neorv32_cache            entities 2, ports   46, signals  40
   neorv32_cpu              entities 1, ports   49, signals  66
   neorv32_cpu_cp_cfu       entities 1, ports   16, signals  13
   neorv32_cpu_cp_muldiv    entities 1, ports   46, signals  24
   neorv32_cpu_pmp          entities 1, ports   45, signals  1

In [2]:
# Step 1b: occurrence profiles with the v3d profiler (rulebook c748e39bfc05 + code facts in validate's CODE CHECK)
# PROFILE_ON: "none" plans only | one module name | "all"
PROFILE_ON = "none"

PROF = S1.Profiler(TEST15)
todo = TEST15 if PROFILE_ON == "all" else [m for m in TEST15 if m == PROFILE_ON]
if todo:
    print("cost of this run:", PROF.run(todo))
PROF.plan()
print("profiles in", S1.PROFILES)


   profiler loaded -- prompts and rulebook from occurrence_prompts_v3d, program code from the v3b builder's cells: self-tests PASS; rulebook c748e39bfc05 (approved); prompts {'classify': '97ef25455d47', 'validate': '021e62bcd902'} (as pinned); 0 warning line(s) []
   copied from occurrence_profiles_v3b: 0 file(s)
rulebook c748e39bfc05: 21 SITE rules, 16 conflict rules, 114 basis quotes
   SITEs added by the rulebook, not in v2: ['CASE_EXPR', 'CASE_COND', 'ASSOC_ACTUAL', 'VAR_RHS_OPERAND', 'INDEXED_NAME', 'ATTR_PREFIX']
   classify prompt 97ef25455d47 (63,784 chars), validate prompt 021e62bcd902 (64,839 chars)
  extract   0 batch(es) to build by code, 117 current on disk, 0 waiting on an earlier step
  classify  81 call(s) to make, 36 current on disk, 0 waiting on an earlier step
     done  neorv32_bus/neorv32_bus_gateway batch 1
     done  neorv32_bus/neorv32_bus_reg batch 1
     done  neorv32_bus/neorv32_bus_reg batch 0
     done  neorv32_bus/neorv32_bus_gateway batch 2
     done  neo

In [3]:
# Step 1b checks: the profiler's own check report (its parse cell) for every module whose profile is complete
ready = [m for m in TEST15 if S1.profile_complete(m)[0]]
print("modules with complete profiles:", ready or "none yet")
if ready:
    PROF.report(ready)


modules with complete profiles: ['neorv32_bus', 'neorv32_cache', 'neorv32_cpu', 'neorv32_cpu_cp_cfu', 'neorv32_cpu_cp_muldiv', 'neorv32_cpu_pmp', 'neorv32_debug_dtm', 'neorv32_hwspinlock', 'neorv32_imem', 'neorv32_spi', 'neorv32_sys', 'neorv32_trng', 'neorv32_twi', 'neorv32_uart', 'neorv32_wdt']
===== neorv32_bus =====
  extract: 870 element profiles, 1640 entries
     entries on a line where the element is not written:   0 of 1640  (exact)
     entries on a line the extract rules exclude:          0 of 1640  (exact)
     occurrences with no entry:                            0 of 1640  (exact under the check's rules)
       of those, on a line that has some entry (merged):   0
  classify: 870 element profiles, 1640 entries
     entries on a line where the element is not written:   0 of 1640  (exact)
     entries on a line the extract rules exclude:          0 of 1640  (exact)
     occurrences with no entry:                            0 of 1640  (exact under the check's rules)
       of

In [7]:
# Step 1c: relationships from the relation annotator, on the Step 1 profiles
# RELATE_ON: "none" plans only | one module name | "all" (only modules whose profiles are complete)
RELATE_ON = "all"
WORKERS   = 4

from dotenv import load_dotenv
from openai import OpenAI
load_dotenv("API.env")
assert RS.audit_prompt(), "the relation prompt fails the audit: stop here"
rel15_client = OpenAI().with_options(timeout=900, max_retries=0)
rel15_cost, rel15_res = {}, {}
for m in TEST15:
    ok, n, miss = S1.profile_complete(m)
    if not ok:
        print(f"   {m:24s} not sent: profile not complete ({n} occurrences, {miss} without a validated SITE)")
        continue
    if RELATE_ON not in ("all", m):
        c = RS.run(m, dry=True, where=S1.WHERE, log=lambda *a: None)["cost"]
        print(f"   {m:24s} {c.get('calls', 0)} call(s) to make, input {c.get('input_chars', 0):,} chars")
        continue
    print(f"\n=== {m}")
    rel15_cost[m] = RS.run(m, client=rel15_client, where=S1.WHERE, workers=WORKERS)["cost"]
    rel15_res[m] = RS.check(m, where=S1.WHERE)
    RS.merge(m, rel15_res[m], where=S1.WHERE)
for m, c in rel15_cost.items():
    print(f"{m}: {c.get('calls', 0)} calls, tokens in {c.get('in', 0):,} out {c.get('out', 0):,}, "
          f"cost at most ${c.get('usd_upper', 0):.4f}")
print("final maps in", S1.REL_MAP)


   prompt relation_system_prompt.md: 27,214 chars, sha a44eebd83bf7; audit PASS

=== neorv32_bus
   current  neorv32_bus_switch b0: 19 elements, 68 occurrences
   current  neorv32_bus_switch b1: 20 elements, 79 occurrences
   current  neorv32_bus_switch b2: 18 elements, 76 occurrences
   current  neorv32_bus_switch b3: 5 elements, 48 occurrences
   current  neorv32_bus_reg b0: 20 elements, 47 occurrences
   current  neorv32_bus_reg b1: 16 elements, 21 occurrences
   current  neorv32_bus_gateway b0: 20 elements, 47 occurrences
   current  neorv32_bus_gateway b1: 20 elements, 22 occurrences
   current  neorv32_bus_gateway b2: 20 elements, 22 occurrences
   current  neorv32_bus_gateway b3: 20 elements, 22 occurrences
   current  neorv32_bus_gateway b4: 17 elements, 81 occurrences
   current  neorv32_bus_gateway b5: 3 elements, 17 occurrences
   current  neorv32_bus_io_switch b0: 20 elements, 25 occurrences
   current  neorv32_bus_io_switch b1: 20 elements, 22 occurrences
   current  neorv

## Relation annotator experiments E1-E4

One change per experiment; each writes its own folder under `step1/lasset_step1/relation_exp/`, so no run overwrites another. Every run appends its numbers to **`step1/lasset_step1/RELATION_EXPERIMENTS_LOG.md`**; the analysis is added there after each run.

| exp | change | modules | calls |
|---|---|---|---|
| E0 | baseline: prompt v1 `a44eebd83bf7`, declaration-order batches (already run: `relation_out/`) | 15 | 117 |
| E1 | nothing: E0 again, to measure run-to-run noise | cache, spi, uart | 23 |
| E2 | batches that keep linked elements together (same caps: 20 elements, 85 occurrences) | 15 | 113 |
| E3 | prompt v2 `8406f390b729`: two-pass pair-first procedure, examples E-G | 15 | 117 |
| E4 | repair pass: each batch holding an unpartnered record is re-called with those records and the other call's side | 15 | 78 on E0 |

**Speed.** All calls of a run, from every module, share one pool of `WORKERS` parallel calls (set in the setup cell, default 20), largest input first. From E0's measured call times a 117-call run takes about 11 minutes at 20 (45 at 4); E4 on E0 (78 calls) about 9. The account limit is 4M tokens per minute; 20 calls stay near 1.1M. A call that fails after its retries is reported and skipped; running the cell again makes only the missing calls.

**Run order.** Run the setup cell first (it reloads the code). E1 and E2 are independent: set `RUN_E1 = True` and `RUN_E2 = True` and run both. E3 and E4 wait for the analysis of the earlier runs: I will say which `BATCHING_E3` and `BASE_E4` to set. With a switch `False`, a cell only prints what it would call and the numbers already on disk.


In [1]:
# Experiments: setup (reloads the code) and the E0 baseline, read from the saved answers (no call)
import importlib, sys
sys.path.insert(0, "step1")
import relation_stage as RS, lasset_step1 as S1, relation_experiments as RX
for _m in (RS, S1, RX):
    importlib.reload(_m)
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv("API.env")
exp_client = OpenAI().with_options(timeout=900, max_retries=0)
TEST15 = S1.test_modules()
E0 = RX.summarize(TEST15, RX.where(None))
RX.show("E0 baseline, all 15 modules", E0)
print("log:", RX.LOG)


WORKERS = 20          # parallel calls in one pool across all modules (about 11 min for 117 calls)


def run_exp(modules, w, batching="declaration", prompt_file=None, workers=WORKERS):
    """Call what is missing (all modules in one pool, largest first), then check and merge every module into the
    experiment's own folders."""
    cost = RS.run_many(modules, client=exp_client, where=w, workers=workers, batching=batching,
                       prompt_file=prompt_file)["cost"]
    print(f"   calls {cost.get('calls', 0)}, failed {cost.get('failed', 0)}, {cost.get('minutes', 0)} min, "
          f"<= ${cost.get('usd_upper', 0)}")
    ps = RX.psha(prompt_file or RX.PROMPT_V1)
    for m in modules:
        RS.merge(m, RS.check(m, psha=ps, where=w, log=lambda *a: None), where=w, log=lambda *a: None)
    return cost.get("calls", 0)


def planned(modules, w, batching="declaration", prompt_file=None):
    return RS.run_many(modules, dry=True, where=w, batching=batching, prompt_file=prompt_file,
                       log=lambda *a: None)["cost"].get("calls", 0)


   E0 baseline, all 15 modules: MIRRORED TYPED PAIRS recall 68.4% precision 94.7%; records 3935, unpartnered 584 (14.8%: omission 280, type disagreement 255, wrong side 49); element pairs R 96.5% P 95.5%; typed R 79.1% P 90.8%; pairs in one call 26%; check failures 39; 117 calls, at most $4.58
log: E:\jobs\ff\test\Prepoison_subset\step1\lasset_step1\RELATION_EXPERIMENTS_LOG.md


In [2]:
# E1 noise floor: the E0 setup, unchanged, on cache, spi and uart (23 calls, about $1.1)
RUN_E1 = True
W1 = RX.where("E1_noise")
print("E1 calls still to make:", planned(RX.NOISE_MODULES, W1))
if RUN_E1:
    made = run_exp(RX.NOISE_MODULES, W1)
    lines = []
    RX.show("E1, second run", RX.summarize(RX.NOISE_MODULES, W1), log=lines.append)
    RX.show("E0 on the same modules", RX.summarize(RX.NOISE_MODULES, RX.where(None)), log=lines.append)
    d = RX.compare_runs(RX.NOISE_MODULES, RX.where(None), W1)
    lines.append(f"   E0 vs E1, same prompt and batches: records {d['records a']} / {d['records b']}, differing "
                 f"{d['records differing']}; element pairs {d['pairs a']} / {d['pairs b']}, differing {d['pairs differing']}")
    print("\n".join(lines))
    if made:
        RX.log_entry("E1 noise floor", "E0 setup again (prompt v1, declaration-order batches) on cache, spi, uart. "
                     "Folder: `relation_exp/E1_noise/`.", lines)


E1 calls still to make: 23
   calling  7 batch(es), 4 at a time ...
   done     neorv32_cache b1: 20 elements, 72 occurrences, 92 s, status completed, out 9,335 tokens
   done     neorv32_cache b0: 20 elements, 83 occurrences, 110 s, status completed, out 11,708 tokens
   done     neorv32_cache b2: 12 elements, 84 occurrences, 119 s, status completed, out 12,778 tokens
   done     neorv32_cache b3: 8 elements, 84 occurrences, 131 s, status completed, out 13,841 tokens
   done     neorv32_cache b4: 4 elements, 13 occurrences, 61 s, status completed, out 6,346 tokens
   done     neorv32_cache_memory b1: 2 elements, 13 occurrences, 44 s, status completed, out 4,732 tokens
   done     neorv32_cache_memory b0: 20 elements, 78 occurrences, 142 s, status completed, out 17,155 tokens
   calling  7 batch(es), 4 at a time ...
   done     neorv32_spi b1: 11 elements, 64 occurrences, 93 s, status completed, out 9,682 tokens
   done     neorv32_spi b2: 5 elements, 85 occurrences, 135 s, status comp

In [3]:
# E2 batching: batches that keep linked elements together; prompt v1 unchanged (113 calls, about $4.5)
RUN_E2 = True
W2 = RX.where("E2_linked")
print("E2 calls still to make:", planned(TEST15, W2, batching="linked"))
if RUN_E2:
    made = run_exp(TEST15, W2, batching="linked")
    lines = []
    RX.show("E2, linked batches", RX.summarize(TEST15, W2), log=lines.append)
    RX.show("E0, declaration-order batches", E0, log=lines.append)
    print("\n".join(lines))
    if made:
        RX.log_entry("E2 linked batches", "Prompt v1 `a44eebd83bf7`; batches from relation_stage.linked_batches (same "
                     "caps). Folder: `relation_exp/E2_linked/`.", lines)


E2 calls still to make: 113
   calling  47 batch(es), 4 at a time ...
   done     neorv32_bus_switch b3: 12 elements, 66 occurrences, 63 s, status completed, out 7,326 tokens
   done     neorv32_bus_switch b2: 20 elements, 45 occurrences, 94 s, status completed, out 11,217 tokens
   done     neorv32_bus_switch b1: 20 elements, 75 occurrences, 96 s, status completed, out 11,435 tokens
   done     neorv32_bus_reg b1: 16 elements, 16 occurrences, 35 s, status completed, out 3,640 tokens
   done     neorv32_bus_reg b0: 20 elements, 52 occurrences, 73 s, status completed, out 8,596 tokens
   done     neorv32_bus_gateway b2: 20 elements, 22 occurrences, 36 s, status completed, out 4,046 tokens
   done     neorv32_bus_gateway b1: 20 elements, 34 occurrences, 64 s, status completed, out 7,235 tokens
   done     neorv32_bus_switch b0: 10 elements, 85 occurrences, 209 s, status completed, out 24,953 tokens
   done     neorv32_bus_gateway b3: 20 elements, 22 occurrences, 53 s, status completed, o

In [2]:
# E3 prompt v2 (two-pass pair-first procedure, examples E-G). BATCHING_E3 is set after the E2 analysis.
RUN_E3 = True
BATCHING_E3 = "declaration"            # "linked" if E2 wins
W3 = RX.where(f"E3_pairfirst_{BATCHING_E3}")
RS.audit_prompt(prompt_file=RX.PROMPT_V2)
print("E3 calls still to make:", planned(TEST15, W3, batching=BATCHING_E3, prompt_file=RX.PROMPT_V2))
if RUN_E3:
    made = run_exp(TEST15, W3, batching=BATCHING_E3, prompt_file=RX.PROMPT_V2, workers=WORKERS)
    base = RX.where(None) if BATCHING_E3 == "declaration" else RX.where("E2_linked")
    lines = []
    RX.show(f"E3, prompt v2, {BATCHING_E3} batches", RX.summarize(TEST15, W3, RX.PROMPT_V2), log=lines.append)
    RX.show(f"same batching with prompt v1", RX.summarize(TEST15, base), log=lines.append)
    print("\n".join(lines))
    if made:
        RX.log_entry("E3 pair-first prompt", f"Prompt v2 `{RX.psha(RX.PROMPT_V2)}`; {BATCHING_E3} batches. Folder: "
                     f"`relation_exp/E3_pairfirst_{BATCHING_E3}/`.", lines)


   prompt relation_prompts_v2/relation_system_prompt.md: 29,103 chars, sha 8406f390b729; audit PASS
E3 calls still to make: 117
   calling  117 batch(es) from 15 module(s), 20 at a time, largest first ...
   done   1/117  bus/neorv32_bus_io_switch b26: 20 elements, 22 occurrences, 47 s, out 3,647 tokens, 0.8 min
   done   2/117  bus/neorv32_bus_io_switch b25: 20 elements, 22 occurrences, 47 s, out 3,755 tokens, 0.8 min
   done   3/117  bus/neorv32_bus_io_switch b9: 20 elements, 22 occurrences, 47 s, out 3,747 tokens, 0.8 min
   done   4/117  bus/neorv32_bus_io_switch b4: 20 elements, 22 occurrences, 48 s, out 3,790 tokens, 0.8 min
   done   5/117  bus/neorv32_bus_io_switch b21: 20 elements, 22 occurrences, 53 s, out 3,666 tokens, 0.9 min
   done   6/117  bus/neorv32_bus_io_switch b20: 20 elements, 22 occurrences, 53 s, out 4,218 tokens, 0.9 min
   done   7/117  bus/neorv32_bus_io_switch b3: 20 elements, 22 occurrences, 54 s, out 3,979 tokens, 0.9 min
   done   8/117  bus/neorv32_bus_io

In [3]:
# E4 repair pass on the best earlier run. BASE_E4 is set after the E3 analysis.
RUN_E4 = True
BASE_E4 = "E0"                          # "E0", "E2" or "E3"
_base = {"E0": (RX.where(None), RX.PROMPT_V1, "declaration"),
         "E2": (RX.where("E2_linked"), RX.PROMPT_V1, "linked"),
         "E3": (RX.where(f"E3_pairfirst_{BATCHING_E3}"), RX.PROMPT_V2, BATCHING_E3)}[BASE_E4]
W4 = RX.where(f"E4_repair_on_{BASE_E4}")
RX.run_repair(TEST15, _base[0], W4, _base[1], exp_client, batching=_base[2], dry=True, workers=WORKERS)
if RUN_E4:
    cost = RX.run_repair(TEST15, _base[0], W4, _base[1], exp_client, batching=_base[2], workers=WORKERS)
    print(f"   repair calls {cost.get('calls', 0)}, failed {cost.get('failed', 0)}, {cost.get('minutes', 0)} min")
    for m in TEST15:
        RS.merge(m, RS.check(m, psha=RX.psha(_base[1]), where=W4, log=lambda *a: None), where=W4, log=lambda *a: None)
    lines = []
    RX.show(f"E4, repaired {BASE_E4}", RX.summarize(TEST15, W4, _base[1]), log=lines.append)
    RX.show(f"{BASE_E4} before repair", RX.summarize(TEST15, _base[0], _base[1]), log=lines.append)
    lines.append(f"   repair calls: {cost}")
    print("\n".join(lines))
    if cost.get("calls"):
        RX.log_entry("E4 repair pass", f"Repair of {BASE_E4}: each batch holding an unpartnered record re-called with "
                     f"the items appended. Folder: `relation_exp/E4_repair_on_{BASE_E4}/`.", lines)


   repair: 78 batch(es) to re-call, 1068 item lines in all
   repair: 78 batch(es) to re-call, 1068 item lines in all
   calling  78 batch(es) from 13 module(s), 20 at a time, largest first ...
   done   1/78  bus/neorv32_bus_io_switch b0: 20 elements, 25 occurrences, 45 s, out 3,672 tokens, 0.7 min
   done   2/78  bus/neorv32_bus_io_switch b4: 20 elements, 22 occurrences, 56 s, out 4,193 tokens, 0.9 min
   done   3/78  bus/neorv32_bus_io_switch b8: 20 elements, 22 occurrences, 56 s, out 4,474 tokens, 0.9 min
   done   4/78  bus/neorv32_bus_io_switch b25: 20 elements, 22 occurrences, 58 s, out 4,622 tokens, 1.0 min
   done   5/78  bus/neorv32_bus_io_switch b18: 20 elements, 22 occurrences, 59 s, out 4,962 tokens, 1.0 min
   done   6/78  bus/neorv32_bus_io_switch b16: 20 elements, 23 occurrences, 60 s, out 4,793 tokens, 1.0 min
   done   7/78  bus/neorv32_bus_io_switch b13: 20 elements, 22 occurrences, 65 s, out 5,074 tokens, 1.1 min
   done   8/78  bus/neorv32_bus_io_switch b6: 20 elem